# Which layers hold redundant information?

This notebook reads the activation caches written by
`scripts/run_activation_caching_early_selected_acts.sh` (`layer_out/0` … `layer_out/16`, GCS tag
`early_`) and `scripts/run_activation_caching_expanded_selected_acts.sh` (`layer_out/17` …
`layer_out/35`, tag `expanded_`), **joins them into one 0-35 sweep per dataset**, and asks
**which layers are near-duplicates of each other**, so downstream work can keep one layer per
redundant group instead of all thirty-six.

The two ranges are cached by separate runs over the same generated prompts, so batch *n* holds
the same rows in both folders; staging checks that their `sample_indices` agree before merging a
batch, and refuses the batch if they do not.

Every metric is computed **per dataset** and then **aggregated across datasets**. That split is
the point: a redundant block appearing in one dataset may be a property of that phrasing, while
a block surviving in all of them is a property of the model and is safe to prune against.

Five matrices, each a different sense of "redundant":

| Matrix | Question it answers |
|---|---|
| **Linear CKA** | Do two layers induce the same geometry over prompts? |
| **Linear predictivity (R²)** | Can layer *j* be linearly reconstructed from layer *i*? (asymmetric) |
| **Subspace overlap** | Do the dominant PCA subspaces of two layers span the same directions? |
| **PLS-1 correlation** | Do two layers order prompts the same way along the time-horizon axis? |
| **Residual CKA** | After the 6 PLS components are removed, is the *remaining* structure still shared? |

Target decodability is reported for **both cached positions** and under **two splits** — an
i.i.d. random split and a deliberate leave-task-out split, where whole tasks are held out — so
generalization across tasks is a number you chose to look at rather than an artifact of row
ordering.

Activations are staged as **on-disk memory-mapped `.npy` arrays**, one per
(dataset, layer, position), and each batch - both ranges of it - is downloaded, consumed, and
deleted before the next, so neither RAM nor disk has to hold a whole folder.

The `conversational` folder (`expanded_selected_acts`) is left out of `DATASETS`: it is the
largest cache by a wide margin, and `conversational_no_output_format` exercises the same prompt
construction. Add it back to the list to include it.


## 1. Setup

For a fresh Colab runtime, uncomment the clone line.


In [ ]:
# !git clone -b dev https://github.com/justinshenk/temporal-manifolds.git
# %cd temporal-manifolds
# !mv -n .env.example .env


In [ ]:
import gc
import json
import math
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.figure_factory as ff
import plotly.graph_objects as go
import torch
from dotenv import load_dotenv
from google.cloud import storage
from numpy.lib.format import open_memmap
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
load_dotenv(repo_root / '.env')


## 2. Authenticate to Google Cloud

In Colab, `google.colab.auth.authenticate_user()` installs the Application Default Credentials
the storage client needs. Without it the client falls back to the Compute Engine metadata
service and fails with a `RefreshError` wrapping a 404 from `metadata.google.internal`. Locally
the credentials come from `gcloud auth application-default login`; this cell reports which it
found rather than running that command for you.


In [ ]:
import google.auth

try:
    from google.colab import auth as colab_auth
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    colab_auth.authenticate_user()
    print('Authenticated through Colab.')

try:
    credentials, detected_project = google.auth.default(
        scopes=['https://www.googleapis.com/auth/devstorage.read_only']
    )
except google.auth.exceptions.DefaultCredentialsError as error:
    raise RuntimeError(
        'No Google Cloud credentials found. In Colab run '
        '`from google.colab import auth; auth.authenticate_user()`; '
        'locally run `gcloud auth application-default login`.'
    ) from error

PROJECT_ID = os.getenv('GCP_PROJECT_ID') or detected_project or 'temporal-interp-exp'
print(f'Credentials: {type(credentials).__name__}')
print(f'Project: {PROJECT_ID}')


## 3. Configure

`DATASETS` names the scenarios written by `scripts/cache_expanded_selected_acts.py`, and
`LAYER_RANGES` names the GCS tag each layer range was uploaded under. A dataset's folder for one
range is simply `tag + suffix`, e.g. `early_abstract_selected_acts` and
`expanded_abstract_selected_acts`. Drop a range from the list to analyze one on its own.

`task_only` declares no time horizon in its prompts, so its target-dependent metrics are skipped
automatically while it still contributes to the geometry matrices.

**Sampling is by batch file, and this is the setting that decides the runtime.** Every batch file
holds 128 prompts and roughly 22-25 MB per layer range, so taking a token quota from *every* file
means downloading a folder in full to keep a handful of rows per file — for the conversational
cache that is 2,618 files, about 124 GB, to keep 2,618 rows. `BATCH_SAMPLE_SIZE` instead picks that
many batch files, **evenly spaced across the folder**, and takes a larger quota from each: the rows
still span the whole dataset (batch *n* holds a contiguous slice of the generated prompts) while the
transfer drops by one to two orders of magnitude. Raise it for a finer spread, set it to `None` to
take every file.

`MAX_BATCH_FILES` truncates the folder listing before sampling and exists for quick development runs;
`MAX_ROWS` is the row count actually analyzed, split evenly across the sampled files.

Two costs scale with `MAX_ROWS`: the CKA step holds one `MAX_ROWS x MAX_ROWS` Gram per layer
(36 layers at 2,000 rows ≈ 580 MB, and it is squared in the row count), and Gram construction
dominates the compute. Lower it to about 1,500 on a memory-tight runtime.


In [ ]:
BUCKET_NAME = 'temporal-research-bucket'


@dataclass(frozen=True)
class Dataset:
    name: str
    gcs_suffix: str        # Folder name without the layer-range tag.


@dataclass(frozen=True)
class LayerRange:
    name: str
    prefix_tag: str        # --prefix-tag passed to the caching script.
    first_layer: int
    last_layer: int


# Mirrors SCENARIOS in scripts/cache_expanded_selected_acts.py.
# 'conversational' (…selected_acts) is deliberately excluded: it is by far the largest
# folder, and its no-output-format variant covers the same prompt construction.
DATASETS = [
    Dataset('conversational_no_output_format', 'NOF_selected_acts'),
    Dataset('abstract', 'abstract_selected_acts'),
    Dataset('plain_english', 'plain_english_selected_acts'),
    Dataset('plain_long', 'plain_long_selected_acts'),
    Dataset('task_only', 'task_only_selected_acts'),
]

# Both caching runs, joined into a single 0-35 sweep. Drop an entry to analyze one range.
LAYER_RANGES = [
    LayerRange('early', 'early_', 0, 16),
    LayerRange('expanded', 'expanded_', 17, 35),
]
REQUIRE_ALL_RANGES = False   # True raises when a dataset is missing a range; False analyzes what exists.


def folder_prefix(dataset, layer_range):
    """GCS folder holding one dataset's activations for one layer range."""
    return f'{layer_range.prefix_tag}{dataset.gcs_suffix}'

DATA_DIR = repo_root / 'data' / 'layer_redundancy'
BATCH_DIR = DATA_DIR / 'batches'
MEMMAP_DIR = DATA_DIR / 'memmaps'

MAX_BATCH_FILES: int | None = None    # Truncate the folder listing before sampling; None keeps all.
BATCH_SAMPLE_SIZE: int | None = 64    # Batch files actually downloaded, evenly spaced; None takes all.
MAX_ROWS = 2_000                      # Rows analyzed per dataset, split evenly across those files.
DOWNLOAD_WORKERS = 8                  # Batch files fetched concurrently.
REBUILD_MEMMAPS = False               # Reuse staged memmaps across notebook restarts.
RANDOM_SEED = 0

PLS_COMPONENTS = 6                    # Matches scripts/fit_expanded_pls_residual_pca.py.
PCA_COMPONENTS = 64                   # Reduced space for linear predictivity.
SUBSPACE_COMPONENTS = 16              # Top-k subspace compared across layers.
FEATURE_CHUNK = 512                   # Columns per block in the chunked Gram/centering passes.
REDUNDANCY_CKA_THRESHOLD = 0.95       # CKA above this counts as redundant when clustering.

# Splits. 'iid' is a shuffled random half; 'leave_task_out' holds out whole tasks, so the
# test half contains no task seen in training.
GROUP_FIELD = 'task'
LEAVE_OUT_GROUP_FRACTION = 0.5
SPLIT_NAMES = ['iid', 'leave_task_out']
RANKING_SPLIT = 'leave_task_out'      # Preferred split for ranking layers; falls back to iid.

rng = np.random.default_rng(RANDOM_SEED)
for directory in (BATCH_DIR, MEMMAP_DIR):
    directory.mkdir(parents=True, exist_ok=True)
client = storage.Client(project=PROJECT_ID, credentials=credentials)
print(f'{len(DATASETS)} dataset folder(s) configured; staging under {DATA_DIR}')


### Palette

One hue, light → dark, for every magnitude matrix (CKA, predictivity, overlap); a blue↔red
diverging pair with a neutral gray midpoint where the sign is meaningful (correlations and
differences); the fixed categorical order for per-dataset line series, assigned by dataset so a
series keeps its color in every chart. Aggregate curves are drawn in ink, not in a categorical
hue, so "mean across datasets" never reads as another dataset. No rainbow scales — a rainbow
makes equal steps in value look like unequal steps in color.


In [ ]:
SEQUENTIAL_BLUE = [
    [0.00, '#f4f8fe'], [0.15, '#cde2fb'], [0.30, '#9ec5f4'], [0.45, '#6da7ec'],
    [0.60, '#3987e5'], [0.75, '#256abf'], [0.90, '#184f95'], [1.00, '#0d366b'],
]
DIVERGING_BLUE_RED = [
    [0.00, '#0d366b'], [0.20, '#256abf'], [0.40, '#9ec5f4'], [0.50, '#f0efec'],
    [0.60, '#f6b0af'], [0.80, '#e34948'], [1.00, '#8f1f1e'],
]
SERIES_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300']
TEXT_PRIMARY, TEXT_SECONDARY, GRID = '#0b0b0b', '#52514e', '#e6e5e1'
DATASET_COLORS = {
    dataset.name: SERIES_COLORS[index % len(SERIES_COLORS)]
    for index, dataset in enumerate(DATASETS)
}


def style_figure(fig, title, subtitle=None, height=480):
    """Apply the shared chart frame: recessive axes, quiet grid, text-token labels."""
    heading = title if subtitle is None else f'{title}<br><sup>{subtitle}</sup>'
    fig.update_layout(
        title={'text': heading, 'font': {'size': 17, 'color': TEXT_PRIMARY}, 'x': 0, 'xanchor': 'left'},
        template='simple_white',
        height=height,
        margin={'l': 70, 'r': 30, 't': 90, 'b': 60},
        font={'color': TEXT_SECONDARY, 'size': 12},
        legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'xanchor': 'left', 'x': 0},
    )
    fig.update_xaxes(showgrid=False, linecolor=GRID, ticks='outside', tickcolor=GRID)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, ticks='outside', tickcolor=GRID)
    return fig


def matrix_figure(matrix, labels, title, subtitle, colorbar_title, *, diverging=False, zmin=None, zmax=None):
    """Render one matrix as a heatmap with per-cell hover."""
    limit = float(np.nanmax(np.abs(matrix))) if diverging else None
    fig = go.Figure(
        go.Heatmap(
            z=matrix,
            x=labels,
            y=labels,
            colorscale=DIVERGING_BLUE_RED if diverging else SEQUENTIAL_BLUE,
            zmid=0 if diverging else None,
            zmin=-limit if diverging else zmin,
            zmax=limit if diverging else zmax,
            colorbar={'title': {'text': colorbar_title, 'side': 'right'}, 'thickness': 14, 'outlinewidth': 0},
            hovertemplate='row %{y}<br>column %{x}<br>value %{z:.3f}<extra></extra>',
        )
    )
    style_figure(fig, title, subtitle, height=620)
    fig.update_yaxes(autorange='reversed', showgrid=False)
    fig.update_layout(width=700)
    return fig


def small_multiple_matrices(matrices, labels, title, subtitle, colorbar_title, *, columns=3, zmin=None, zmax=None):
    """Render one heatmap per dataset on a shared color axis."""
    names = list(matrices)
    rows = int(np.ceil(len(names) / columns))
    fig = make_subplots(
        rows=rows,
        cols=columns,
        subplot_titles=names,
        horizontal_spacing=0.08,
        vertical_spacing=0.12,
    )
    for index, name in enumerate(names):
        fig.add_trace(
            go.Heatmap(
                z=matrices[name],
                x=labels,
                y=labels,
                coloraxis='coloraxis',
                hovertemplate=f'{name}<br>row %{{y}}<br>column %{{x}}<br>value %{{z:.3f}}<extra></extra>',
            ),
            row=index // columns + 1,
            col=index % columns + 1,
        )
    style_figure(fig, title, subtitle, height=320 * rows + 120)
    fig.update_layout(
        coloraxis={
            'colorscale': SEQUENTIAL_BLUE,
            'cmin': zmin,
            'cmax': zmax,
            'colorbar': {'title': {'text': colorbar_title, 'side': 'right'}, 'thickness': 14, 'outlinewidth': 0},
        }
    )
    fig.update_yaxes(autorange='reversed', showgrid=False)
    fig.update_xaxes(showgrid=False)
    for annotation in fig.layout.annotations:
        annotation.font.size = 12
        annotation.font.color = TEXT_PRIMARY
    return fig


def line_figure(x_values, series, title, subtitle, y_title, x_title='Layer', emphasis=None, colors=None):
    """Per-dataset curves in categorical hues; an optional aggregate curve drawn in ink."""
    palette = colors if colors is not None else DATASET_COLORS
    fig = go.Figure()
    for index, (name, values) in enumerate(series.items()):
        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=values,
                name=name,
                mode='lines+markers',
                line={'width': 2, 'color': palette.get(name, SERIES_COLORS[index % len(SERIES_COLORS)])},
                marker={'size': 7},
                hovertemplate=f'{name}<br>layer %{{x}}<br>%{{y:.3f}}<extra></extra>',
            )
        )
    if emphasis is not None:
        emphasis_name, emphasis_values = emphasis
        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=emphasis_values,
                name=emphasis_name,
                mode='lines+markers',
                line={'width': 3, 'color': TEXT_PRIMARY, 'dash': 'dot'},
                marker={'size': 8, 'symbol': 'diamond'},
                hovertemplate=f'{emphasis_name}<br>layer %{{x}}<br>%{{y:.3f}}<extra></extra>',
            )
        )
    style_figure(fig, title, subtitle, height=500)
    fig.update_layout(showlegend=len(series) + (emphasis is not None) > 1)
    fig.update_xaxes(title_text=x_title, dtick=1)
    fig.update_yaxes(title_text=y_title)
    return fig


## 4. Stage each dataset into memory-mapped arrays

One streaming pass per dataset over the *sampled* batch files. Files are fetched
`DOWNLOAD_WORKERS` at a time so the network is not idle between batches, then each group is staged
serially and deleted before the next group is fetched - peak disk stays at the memmaps plus one
group of `.pt` files. For each batch, every configured layer range's copy is checked against the
others' `sample_indices` before their layers are merged, and the same sampled rows are copied out
of each into one `.npy` memmap per (layer, position). The memmaps are then centered in place over
column chunks, and a small `manifest.json` records the row count, joined layer list, positions, and
target so a later run can reuse the staged arrays without touching the network.


In [ ]:
UNIT_TO_MONTHS = {
    'second': 1 / (30.4375 * 86400), 'minute': 1 / (30.4375 * 1440),
    'hour': 1 / (30.4375 * 24), 'day': 1 / 30.4375, 'week': 7 / 30.4375,
    'month': 1.0, 'year': 12.0, 'decade': 120.0, 'century': 1200.0, 'millennium': 12000.0,
}
UNIT_TO_MONTHS.update({f'{unit}s': value for unit, value in list(UNIT_TO_MONTHS.items())})
UNIT_TO_MONTHS['centuries'] = 1200.0
UNIT_TO_MONTHS['millennia'] = 12000.0


def horizon_months(metadata):
    """Return a prompt's horizon in months, or None when it declares none."""
    value = metadata.get('base_value', metadata.get('value'))
    unit = metadata.get('base_unit', metadata.get('unit'))
    if value in (None, 'N/A') or unit in (None, 'N/A'):
        return None
    return float(value) * UNIT_TO_MONTHS[str(unit).lower()]


def flatten_metadata(metadata):
    """Flatten one prompt's nested metadata into dotted scalar fields."""
    flattened = {}
    for key, value in metadata.items():
        if isinstance(value, dict):
            flattened.update({f'{key}.{inner}': item for inner, item in value.items()})
        elif not isinstance(value, (list, tuple, set)):
            flattened[key] = value
    return flattened


def list_batch_blobs(prefix):
    """Every activation batch blob below one folder prefix, ordered by name."""
    return sorted(
        (
            blob for blob in client.bucket(BUCKET_NAME).list_blobs(prefix=prefix + '/')
            if Path(blob.name).name.startswith('activations_batch_') and blob.name.endswith('.pt')
        ),
        key=lambda blob: blob.name,
    )


def blobs_by_range(dataset):
    """Return {range name: {batch filename: blob}} for the ranges this dataset has.

    A range with no folder in the bucket is skipped, unless REQUIRE_ALL_RANGES is set.
    """
    available = {}
    for layer_range in LAYER_RANGES:
        prefix = folder_prefix(dataset, layer_range)
        blobs = list_batch_blobs(prefix)
        if not blobs:
            message = f'No activation batches below gs://{BUCKET_NAME}/{prefix}'
            if REQUIRE_ALL_RANGES:
                raise FileNotFoundError(message)
            print(f'  {message}; continuing without the {layer_range.name} range.')
            continue
        available[layer_range.name] = {Path(blob.name).name: blob for blob in blobs}
    if not available:
        raise FileNotFoundError(f'{dataset.name}: no layer range is present in the bucket.')
    return available


def shared_batch_names(available):
    """Batch filenames present in every available range, truncated then evenly sampled.

    Sampling whole files rather than a token quota from each is what keeps the download
    proportional to the rows actually analyzed. Evenly spaced picks preserve coverage,
    because batch *n* holds a contiguous slice of the generated prompt list.
    """
    common = set.intersection(*(set(names) for names in available.values()))
    ordered = sorted(common)
    if not ordered:
        raise FileNotFoundError('The configured layer ranges share no batch files.')
    skipped = sum(len(names) for names in available.values()) - len(ordered) * len(available)
    if skipped:
        print(f'  {skipped} batch file(s) exist in only one range and are skipped.')
    if MAX_BATCH_FILES is not None:
        ordered = ordered[:MAX_BATCH_FILES]
    if BATCH_SAMPLE_SIZE is None or len(ordered) <= BATCH_SAMPLE_SIZE:
        return ordered
    picks = np.linspace(0, len(ordered) - 1, BATCH_SAMPLE_SIZE).round().astype(int)
    sampled = [ordered[position] for position in dict.fromkeys(picks.tolist())]
    print(f'  sampling {len(sampled)} of {len(ordered)} batch files, evenly spaced.')
    return sampled


def download_batch_group(available, range_names, batch_name, download_dir):
    """Fetch one batch file from every range; returns the local paths."""
    paths = {}
    for range_name in range_names:
        local_path = download_dir / f'{range_name}_{batch_name}'
        available[range_name][batch_name].download_to_filename(str(local_path))
        paths[range_name] = local_path
    return batch_name, paths


def memmap_dir(dataset):
    return MEMMAP_DIR / dataset.name


def memmap_path(dataset, layer, position_index):
    return memmap_dir(dataset) / f'layer_{layer:02d}_pos{position_index}.npy'


def load_manifest(dataset):
    """Return a previously staged manifest, or None when staging is needed."""
    directory = memmap_dir(dataset)
    manifest_path = directory / 'manifest.json'
    if REBUILD_MEMMAPS or not manifest_path.exists():
        return None
    manifest = json.loads(manifest_path.read_text())
    paths = {
        (layer, position_index): memmap_path(dataset, layer, position_index)
        for layer in manifest['layers']
        for position_index in range(len(manifest['positions']))
    }
    if not all(path.exists() for path in paths.values()):
        return None
    manifest['dataset'] = dataset
    manifest['paths'] = paths
    manifest['metadata'] = pd.read_csv(directory / 'metadata.csv')
    target_path = directory / 'target.npy'
    manifest['target'] = np.load(target_path) if target_path.exists() else None
    manifest['has_target'] = manifest['target'] is not None
    return manifest


def stage_dataset(dataset):
    """Stream one dataset's batches into centered, memory-mapped activation arrays.

    Every configured layer range contributes its own copy of each batch; the copies are
    checked for row alignment, merged into one array set spanning the joined layer list,
    and deleted before the next batch, so no folder is ever on disk in full. Rows are
    drawn as an equal quota per batch, spreading the sample across the whole dataset
    instead of concentrating it in the first files.
    """
    existing = load_manifest(dataset)
    if existing is not None:
        print(f'{dataset.name}: reusing {existing["row_count"]:,} staged rows in {memmap_dir(dataset)}')
        return existing

    available = blobs_by_range(dataset)
    batch_names = shared_batch_names(available)
    range_names = [layer_range.name for layer_range in LAYER_RANGES if layer_range.name in available]
    print(f'  ranges: {", ".join(range_names)}; batches: {len(batch_names):,}')

    directory = memmap_dir(dataset)
    directory.mkdir(parents=True, exist_ok=True)
    download_dir = BATCH_DIR / dataset.name
    download_dir.mkdir(parents=True, exist_ok=True)

    quota = None if MAX_ROWS is None else max(1, math.ceil(MAX_ROWS / len(batch_names)))
    arrays = None
    layers = positions = None
    hidden_size = 0
    capacity = None
    write_cursor = 0
    metadata_records = []
    horizons = []

    progress = tqdm(total=len(batch_names), desc=f'{dataset.name}: staging', leave=False)
    executor = ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS)
    try:
      for group_start in range(0, len(batch_names), DOWNLOAD_WORKERS):
        group = batch_names[group_start:group_start + DOWNLOAD_WORKERS]
        fetched = list(executor.map(
            lambda name: download_batch_group(available, range_names, name, download_dir), group
        ))
        for batch_name, local_paths in fetched:
            payloads = {}
            try:
                for range_name, local_path in local_paths.items():
                    payloads[range_name] = torch.load(
                        local_path, map_location='cpu', weights_only=True, mmap=True
                    )

                reference = payloads[range_names[0]]
                batch_positions = list(reference['positions'])
                batch_layers = sorted(
                    int(key.split('/')[-1])
                    for payload in payloads.values()
                    for key in payload['activations']
                )
                for range_name, payload in payloads.items():
                    if list(payload['sample_indices']) != list(reference['sample_indices']):
                        raise ValueError(
                            f'{batch_name}: the {range_name} range holds different rows than '
                            f'{range_names[0]}; the two caching runs are not aligned.'
                        )
                    if list(payload['positions']) != batch_positions:
                        raise ValueError(f'{batch_name}: the {range_name} range cached other positions.')
                if len(set(batch_layers)) != len(batch_layers):
                    raise ValueError(f'{batch_name}: the configured ranges overlap on a layer.')

                if layers is None:
                    layers, positions = batch_layers, batch_positions
                    first_range = next(iter(payloads.values()))
                    hidden_size = int(next(iter(first_range['activations'].values())).shape[2])
                    capacity = len(batch_names) * quota if quota is not None else None
                elif batch_layers != layers or batch_positions != positions:
                    raise ValueError(f'{batch_name} disagrees with earlier batches on layers or positions.')

                batch_rows = len(reference['prompt_metadata'])
                if capacity is None:
                    capacity = batch_rows * len(batch_names)
                if arrays is None:
                    arrays = {
                        (layer, position_index): open_memmap(
                            memmap_path(dataset, layer, position_index),
                            mode='w+', dtype=np.float32, shape=(capacity, hidden_size),
                        )
                        for layer in layers
                        for position_index in range(len(positions))
                    }

                take = batch_rows if quota is None else min(quota, batch_rows, capacity - write_cursor)
                if take <= 0:
                    continue
                offsets = np.sort(rng.choice(batch_rows, size=take, replace=False))
                index = torch.as_tensor(offsets.tolist(), dtype=torch.long)
                for payload in payloads.values():
                    for key, activation in payload['activations'].items():
                        layer = int(key.split('/')[-1])
                        tensor = activation.index_select(0, index).to(torch.float32)
                        for position_index in range(len(positions)):
                            arrays[(layer, position_index)][write_cursor:write_cursor + take] = (
                                tensor[:, position_index, :].numpy()
                            )
                        del tensor
                for offset in offsets:
                    metadata = reference['prompt_metadata'][int(offset)]
                    record = flatten_metadata(metadata)
                    record['sample_index'] = int(reference['sample_indices'][int(offset)])
                    record['batch_file'] = batch_name
                    record['time_horizon_months'] = horizon_months(metadata)
                    metadata_records.append(record)
                    horizons.append(record['time_horizon_months'])
                write_cursor += take
            finally:
                payloads.clear()
                for local_path in local_paths.values():
                    local_path.unlink(missing_ok=True)
                progress.update(1)
    finally:
        executor.shutdown(wait=True)
        progress.close()
        gc.collect()

    row_count = write_cursor
    if row_count == 0:
        raise ValueError(f'{dataset.name}: no rows were staged.')

    # Center the used slice in place, over column chunks, so nothing large is held in RAM.
    for array in tqdm(arrays.values(), desc=f'{dataset.name}: centering', leave=False):
        for start in range(0, hidden_size, FEATURE_CHUNK):
            block = array[:row_count, start:start + FEATURE_CHUNK]
            block -= block.mean(axis=0, keepdims=True)
        array.flush()
    del arrays
    gc.collect()
    shutil.rmtree(download_dir, ignore_errors=True)

    metadata_df = pd.DataFrame(metadata_records).reset_index(drop=True)
    has_target = all(value is not None for value in horizons)
    target = np.log10(np.array(horizons, dtype=np.float64)) if has_target else None

    metadata_df.to_csv(directory / 'metadata.csv', index=False)
    if target is not None:
        np.save(directory / 'target.npy', target)
    (directory / 'manifest.json').write_text(json.dumps({
        'dataset': dataset.name,
        'gcs_prefixes': [folder_prefix(dataset, layer_range) for layer_range in LAYER_RANGES
                         if layer_range.name in available],
        'layers': layers, 'positions': positions, 'row_count': row_count,
        'hidden_size': hidden_size, 'batch_files': len(batch_names), 'capacity': int(capacity),
    }, indent=2))

    return {
        'dataset': dataset, 'layers': layers, 'positions': positions,
        'paths': {
            (layer, position_index): memmap_path(dataset, layer, position_index)
            for layer in layers for position_index in range(len(positions))
        },
        'target': target, 'has_target': has_target, 'metadata': metadata_df,
        'row_count': row_count, 'hidden_size': hidden_size,
    }


def open_staged(manifest, layer, position_index):
    """Open one staged activation array read-only, sliced to the staged row count."""
    array = np.load(manifest['paths'][(layer, position_index)], mmap_mode='r')
    return array[:manifest['row_count']]


## 5. Splits

Rows are staged in dataset order, so a positional half-and-half split silently holds out
whichever tasks happen to sort last. Both splits are therefore constructed explicitly:

- **i.i.d.** — a shuffled random half. Measures readout quality on prompts drawn from the same
  distribution as training.
- **leave-task-out** — whole values of `GROUP_FIELD` (`task`) are held out, so no task in the
  test half was seen in training. Measures whether the horizon axis generalizes across tasks.

Both are reported for the target metrics. The predictivity matrix uses the i.i.d. split, since
it asks about reconstructing one representation from another rather than about generalization.


In [ ]:
def build_splits(metadata_df, row_count):
    """Return {'iid': (train, test), 'leave_task_out': (train, test)} where possible."""
    minimum_rows = PLS_COMPONENTS + 1
    splits = {}

    order = rng.permutation(row_count)
    half = row_count // 2
    if half >= minimum_rows and row_count - half >= minimum_rows:
        splits['iid'] = (np.sort(order[:half]), np.sort(order[half:]))

    if GROUP_FIELD in metadata_df.columns:
        groups = metadata_df[GROUP_FIELD].astype(str).to_numpy()
        unique = np.array(sorted({value for value in groups if value not in ('N/A', 'nan')}))
        if len(unique) >= 2:
            shuffled = rng.permutation(unique)
            holdout_count = max(1, int(round(len(unique) * LEAVE_OUT_GROUP_FRACTION)))
            holdout = set(shuffled[:holdout_count].tolist())
            test_mask = np.array([value in holdout for value in groups])
            train_rows = np.flatnonzero(~test_mask)
            test_rows = np.flatnonzero(test_mask)
            if len(train_rows) >= minimum_rows and len(test_rows) >= minimum_rows:
                splits['leave_task_out'] = (train_rows, test_rows)

    if not splits:
        raise ValueError('Too few rows to build any split.')
    return splits


def condition_key(position_value, split_name):
    """Name one (position, split) reporting condition."""
    return f'pos{position_value}_{split_name}'


## 6. Metric definitions

Each function reads memmaps and returns a small matrix or vector. The Gram matrices behind CKA
are accumulated over column chunks, so a layer's activations stream off disk rather than loading
whole.


In [ ]:
def chunked_gram(array):
    """Gram matrix of centered rows, accumulated over feature chunks."""
    rows = array.shape[0]
    gram = np.zeros((rows, rows), dtype=np.float32)
    for start in range(0, array.shape[1], FEATURE_CHUNK):
        block = np.asarray(array[:, start:start + FEATURE_CHUNK], dtype=np.float32)
        gram += block @ block.T
    return gram


def cka_from_grams(gram_a, gram_b, norm_a=None, norm_b=None):
    """Linear CKA from two centered Gram matrices."""
    norm_a = float(np.sqrt((gram_a * gram_a).sum())) if norm_a is None else norm_a
    norm_b = float(np.sqrt((gram_b * gram_b).sum())) if norm_b is None else norm_b
    return float((gram_a * gram_b).sum() / (norm_a * norm_b))


def cka_matrix_from_grams(grams):
    """Linear CKA between every pair of layers, from Gram matrices already built."""
    norms = [float(np.sqrt((gram * gram).sum())) for gram in grams]
    size = len(grams)
    matrix = np.ones((size, size), dtype=np.float64)
    for i in range(size):
        for j in range(i + 1, size):
            matrix[i, j] = matrix[j, i] = cka_from_grams(grams[i], grams[j], norms[i], norms[j])
    return matrix


def build_grams(arrays, description):
    """Gram matrix per layer. Each costs rows^2 x hidden flops, so they are built once and reused."""
    return [chunked_gram(array) for array in tqdm(arrays, desc=description, leave=False)]


def cka_matrix(arrays, description):
    """Linear CKA between every pair of layers."""
    grams = build_grams(arrays, description)
    matrix = cka_matrix_from_grams(grams)
    del grams
    gc.collect()
    return matrix


def linear_predictivity_matrix(reduced, train_rows, test_rows):
    """Held-out R² of predicting each layer's reduced code from every other layer's."""
    size = len(reduced)
    matrix = np.eye(size)
    for i in range(size):
        design = np.column_stack([reduced[i][train_rows], np.ones(len(train_rows))])
        design_test = np.column_stack([reduced[i][test_rows], np.ones(len(test_rows))])
        for j in range(size):
            if i == j:
                continue
            destination = reduced[j]
            coefficients, *_ = np.linalg.lstsq(design, destination[train_rows], rcond=None)
            predicted = design_test @ coefficients
            actual = destination[test_rows]
            residual_ss = float(np.square(actual - predicted).sum())
            total_ss = float(np.square(actual - actual.mean(axis=0)).sum())
            matrix[i, j] = 1.0 - residual_ss / total_ss
    return matrix


def subspace_overlap_matrix(bases):
    """Mean squared cosine of principal angles between leading PCA subspaces."""
    size = len(bases)
    matrix = np.eye(size)
    for i in range(size):
        for j in range(i + 1, size):
            singular = np.linalg.svd(bases[i].T @ bases[j], compute_uv=False)
            matrix[i, j] = matrix[j, i] = float(np.square(singular).mean())
    return matrix


def participation_ratio(eigenvalues):
    """Effective number of directions carrying the variance."""
    return float(np.square(eigenvalues.sum()) / np.square(eigenvalues).sum())


## 7. Process every dataset

One dataset at a time: stage → compute → drop the in-memory intermediates. Geometry matrices use
the final prompt token (position -1), and the other position is summarized by a per-layer CKA
curve between the two. Target R² is computed for **every (position, split) combination**; the
PLS score correlation and residual CKA use a model fitted on all rows at the final position,
since those are descriptive matrices rather than predictions.

Expect this to be the long cell — downloading the folders and building Grams both dominate.


In [ ]:
MATRIX_FIELDS = ('cka', 'predictivity', 'subspace_overlap', 'residual_cka', 'pls_score_correlation')
VECTOR_FIELDS = ('adjacent_cka', 'position_cka', 'participation_ratio')


def restrict_result_to_layers(result, common_layers):
    """Slice one dataset's matrices and curves down to a layer list shared by all datasets.

    Needed when a dataset is missing a layer range: the aggregates stack matrices across
    datasets, so every dataset must contribute the same layers.
    """
    if list(result['layers']) == list(common_layers):
        return result
    keep = [list(result['layers']).index(layer) for layer in common_layers]
    grid = np.ix_(keep, keep)
    for field in MATRIX_FIELDS:
        matrix = result.get(field)
        if matrix is not None:
            result[field] = np.asarray(matrix)[grid]
    for field in VECTOR_FIELDS:
        values = result.get(field)
        if values is not None:
            result[field] = [values[position] for position in keep]
    result['target_r2'] = {
        key: [values[position] for position in keep]
        for key, values in result['target_r2'].items()
    }
    result['layers'] = list(common_layers)
    return result


def analyze_dataset(dataset):
    manifest = stage_dataset(dataset)
    layers = manifest['layers']
    positions = manifest['positions']
    final_position = len(positions) - 1
    row_count = manifest['row_count']
    target = manifest['target']

    component_limit = min(PCA_COMPONENTS, row_count - 1, manifest['hidden_size'])
    subspace_limit = min(SUBSPACE_COMPONENTS, component_limit)
    splits = build_splits(manifest['metadata'], row_count)

    final_arrays = [open_staged(manifest, layer, final_position) for layer in layers]
    result = {
        'dataset': dataset,
        'layers': layers,
        'positions': positions,
        'row_count': row_count,
        'has_target': manifest['has_target'],
        'component_limit': component_limit,
        'subspace_limit': subspace_limit,
        'splits': {name: (len(train), len(test)) for name, (train, test) in splits.items()},
    }

    # The final-position Grams serve both the CKA matrix and the position curve below.
    final_grams = build_grams(final_arrays, f'{dataset.name}: Gram matrices')
    result['cka'] = cka_matrix_from_grams(final_grams)
    result['adjacent_cka'] = [np.nan] + [
        float(result['cka'][index - 1, index]) for index in range(1, len(layers))
    ]

    result['position_cka'] = []
    for layer_offset, layer in enumerate(tqdm(layers, desc=f'{dataset.name}: position CKA', leave=False)):
        gram_first = chunked_gram(open_staged(manifest, layer, 0))
        result['position_cka'].append(cka_from_grams(gram_first, final_grams[layer_offset]))
        del gram_first
    del final_grams
    gc.collect()

    reduced, bases, ratios = [], [], []
    for array in tqdm(final_arrays, desc=f'{dataset.name}: PCA', leave=False):
        pca = PCA(n_components=component_limit, svd_solver='randomized', random_state=RANDOM_SEED)
        reduced.append(pca.fit_transform(np.asarray(array)))
        bases.append(pca.components_[:subspace_limit].T)
        ratios.append(participation_ratio(pca.explained_variance_))
    predictivity_split = splits.get('iid') or next(iter(splits.values()))
    result['predictivity'] = linear_predictivity_matrix(reduced, *predictivity_split)
    result['subspace_overlap'] = subspace_overlap_matrix(bases)
    result['participation_ratio'] = ratios
    del reduced, bases
    gc.collect()

    result['target_r2'] = {}
    result['conditions'] = []
    if manifest['has_target']:
        for position_index, position_value in enumerate(positions):
            scores_by_split = {name: [] for name in splits}
            first_scores, residual_arrays = [], []
            for layer in tqdm(layers, desc=f'{dataset.name}: PLS pos {position_value}', leave=False):
                features = np.asarray(open_staged(manifest, layer, position_index))
                for split_name, (train_rows, test_rows) in splits.items():
                    model = PLSRegression(n_components=PLS_COMPONENTS, scale=False)
                    model.fit(features[train_rows], target[train_rows])
                    scores_by_split[split_name].append(
                        float(model.score(features[test_rows], target[test_rows]))
                    )
                if position_index == final_position:
                    full_model = PLSRegression(n_components=PLS_COMPONENTS, scale=False)
                    full_model.fit(features, target)
                    full_scores = full_model.transform(features)
                    first_scores.append(full_scores[:, 0])
                    residual_arrays.append(features - full_model.inverse_transform(full_scores))
                del features
            for split_name, values in scores_by_split.items():
                key = condition_key(position_value, split_name)
                result['target_r2'][key] = values
                result['conditions'].append(key)
            if position_index == final_position:
                result['pls_score_correlation'] = np.corrcoef(np.column_stack(first_scores), rowvar=False)
                result['residual_cka'] = cka_matrix(residual_arrays, f'{dataset.name}: residual CKA')
                del first_scores, residual_arrays
            gc.collect()
    else:
        result['pls_score_correlation'] = None
        result['residual_cka'] = None

    del final_arrays
    gc.collect()
    return result


results_by_dataset = {}
for dataset in DATASETS:
    folders = ', '.join(folder_prefix(dataset, layer_range) for layer_range in LAYER_RANGES)
    print(f'=== {dataset.name} ({folders}) ===')
    try:
        result = analyze_dataset(dataset)
    except FileNotFoundError as error:
        print(f'  skipped: {error}')
        continue
    results_by_dataset[dataset.name] = result
    split_note = ', '.join(f'{name} {sizes[0]}/{sizes[1]}' for name, sizes in result['splits'].items())
    print(f'  rows {result["row_count"]:,}; layers {result["layers"][0]}..{result["layers"][-1]}; splits (train/test): {split_note}')
    if result['has_target']:
        for key, values in result['target_r2'].items():
            print(f'    best R² {np.nanmax(values):+.3f} at layer {result["layers"][int(np.nanargmax(values))]}  [{key}]')
    else:
        print('    no time horizon in this dataset; target metrics skipped')

if not results_by_dataset:
    raise RuntimeError('No dataset produced results.')

layer_sets = [set(result['layers']) for result in results_by_dataset.values()]
common_layers = sorted(set.intersection(*layer_sets))
if not common_layers:
    raise RuntimeError('The analyzed datasets share no layers; check which ranges are cached.')
missing = sorted(set.union(*layer_sets) - set(common_layers))
if missing:
    print(f'Layers {missing} are absent from at least one dataset and are excluded from every aggregate.')
for result in results_by_dataset.values():
    restrict_result_to_layers(result, common_layers)

layers = common_layers
positions = next(iter(results_by_dataset.values()))['positions']
layer_labels = [str(layer) for layer in layers]
dataset_names = list(results_by_dataset)
target_datasets = [name for name in dataset_names if results_by_dataset[name]['has_target']]
conditions = list(dict.fromkeys(
    key for name in target_datasets for key in results_by_dataset[name]['target_r2']
))
triangle = np.triu_indices(len(layers), k=1)
print(f'Analyzed {len(dataset_names)} dataset(s); {len(target_datasets)} with a time-horizon target.')
print(f'Reporting conditions: {conditions}')


## 8. Per-dataset matrices

All panels share one color axis, so a block that looks brighter in one dataset really is more
redundant there.


In [ ]:
cka_by_dataset = {name: results_by_dataset[name]['cka'] for name in dataset_names}
small_multiple_matrices(
    cka_by_dataset,
    layer_labels,
    'Linear CKA between layers, per dataset',
    f'Position {positions[-1]}; bright diagonal blocks are runs of near-duplicate layers',
    'CKA',
    zmin=float(min(matrix.min() for matrix in cka_by_dataset.values())),
    zmax=1.0,
).show()


In [ ]:
small_multiple_matrices(
    {name: results_by_dataset[name]['predictivity'] for name in dataset_names},
    layer_labels,
    'Linear predictivity R², per dataset',
    'Row = predictor layer, column = predicted layer; i.i.d. split of the reduced PCA codes',
    'R²',
    zmin=0.0,
    zmax=1.0,
).show()


In [ ]:
small_multiple_matrices(
    {name: results_by_dataset[name]['subspace_overlap'] for name in dataset_names},
    layer_labels,
    'Principal-subspace overlap, per dataset',
    'Mean squared cosine of principal angles between the leading PCA subspaces',
    'overlap',
    zmin=0.0,
    zmax=1.0,
).show()


## 9. Aggregated matrices

The mean across datasets is the headline; the spread says how much to trust it. A cell that is
high in the mean **and** low in the standard deviation is redundancy that holds regardless of
phrasing.


In [ ]:
stacked_cka = np.stack([results_by_dataset[name]['cka'] for name in dataset_names])
mean_cka = stacked_cka.mean(axis=0)
std_cka = stacked_cka.std(axis=0)

matrix_figure(
    mean_cka,
    layer_labels,
    'Linear CKA, mean across datasets',
    f'Average of {len(dataset_names)} dataset(s) at position {positions[-1]}',
    'mean CKA',
    zmin=float(mean_cka.min()),
    zmax=1.0,
).show()

matrix_figure(
    std_cka,
    layer_labels,
    'Cross-dataset standard deviation of CKA',
    'Bright cells are layer pairs whose similarity depends on which dataset you look at',
    'std CKA',
    zmin=0.0,
).show()


In [ ]:
stacked_predictivity = np.stack([results_by_dataset[name]['predictivity'] for name in dataset_names])
matrix_figure(
    stacked_predictivity.mean(axis=0),
    layer_labels,
    'Linear predictivity R², mean across datasets',
    'Row predicts column; asymmetry shows which layer is a compression of which',
    'mean R²',
    zmin=0.0,
    zmax=1.0,
).show()

stacked_overlap = np.stack([results_by_dataset[name]['subspace_overlap'] for name in dataset_names])
matrix_figure(
    stacked_overlap.mean(axis=0),
    layer_labels,
    'Principal-subspace overlap, mean across datasets',
    'Leading subspaces that coincide in every dataset are the safest merges',
    'mean overlap',
    zmin=0.0,
    zmax=1.0,
).show()


### Do the datasets agree with each other?

Each dataset's CKA matrix is flattened to its upper triangle and correlated against every other
dataset's. High values mean the redundancy structure is a property of the model rather than of
one dataset's phrasing — which is the licence for pruning layers globally.


In [ ]:
if len(dataset_names) > 1:
    flattened = np.stack([results_by_dataset[name]['cka'][triangle] for name in dataset_names])
    matrix_figure(
        np.corrcoef(flattened),
        dataset_names,
        'Cross-dataset agreement on the CKA structure',
        'Correlation between datasets of the off-diagonal CKA values',
        'r',
        diverging=True,
    ).show()
else:
    print('Only one dataset in this run; the agreement matrix needs at least two.')


### Where the representation changes, by dataset

Flat stretches near 1.0 are the redundant runs; dips mark layers doing real work. The dotted ink
curve is the mean across datasets.


In [ ]:
def stack_series(series):
    return np.stack([np.asarray(values, dtype=float) for values in series.values()])


adjacent_series = {name: results_by_dataset[name]['adjacent_cka'] for name in dataset_names}
mean_adjacent = np.nanmean(stack_series(adjacent_series), axis=0)
line_figure(
    layers,
    adjacent_series,
    'CKA with the previous layer',
    'Values near 1.0 mean the layer adds little the previous layer did not already hold',
    'CKA(layer, layer - 1)',
    emphasis=('mean across datasets', mean_adjacent),
).show()


In [ ]:
position_series = {name: results_by_dataset[name]['position_cka'] for name in dataset_names}
mean_position = np.nanmean(stack_series(position_series), axis=0)
line_figure(
    layers,
    position_series,
    f'Agreement between cached positions {positions[0]} and {positions[-1]}',
    'High values mean the second cached position adds little beyond the final token',
    'CKA between positions',
    emphasis=('mean across datasets', mean_position),
).show()


In [ ]:
participation_series = {name: results_by_dataset[name]['participation_ratio'] for name in dataset_names}
mean_participation = np.nanmean(stack_series(participation_series), axis=0)
line_figure(
    layers,
    participation_series,
    'Effective dimensionality (participation ratio)',
    'From each layer leading PCA spectrum; higher means variance spread over more directions',
    'participation ratio',
    emphasis=('mean across datasets', mean_participation),
).show()


## 10. Redundancy *about the target*

Two layers can be geometrically similar yet differ in how cleanly the time horizon reads out —
that is what decides which member of a redundant group to keep. Every combination of cached
position and split is reported. The gap between the i.i.d. and leave-task-out curves is the
interesting quantity: it says how much of a layer's apparent horizon signal is task-specific.
`task_only` declares no horizon, so it sits out this section.


In [ ]:
mean_target_by_condition = {}
for key in conditions:
    values = [
        np.asarray(results_by_dataset[name]['target_r2'][key], dtype=float)
        for name in target_datasets if key in results_by_dataset[name]['target_r2']
    ]
    mean_target_by_condition[key] = np.nanmean(np.stack(values), axis=0)

if mean_target_by_condition:
    condition_colors = {
        key: SERIES_COLORS[index % len(SERIES_COLORS)] for index, key in enumerate(mean_target_by_condition)
    }
    line_figure(
        layers,
        mean_target_by_condition,
        f'Held-out R² of a {PLS_COMPONENTS}-component PLS on log10_time_horizon_months',
        'Mean across datasets, by cached position and split; the i.i.d. minus leave-task-out gap is task-specific signal',
        'R² (held-out)',
        colors=condition_colors,
    ).show()
else:
    print('No dataset in this run carries a time-horizon target; skipping the target section.')


In [ ]:
for key in conditions:
    series = {
        name: results_by_dataset[name]['target_r2'][key]
        for name in target_datasets if key in results_by_dataset[name]['target_r2']
    }
    if not series:
        continue
    line_figure(
        layers,
        series,
        f'Target R² per dataset — {key}',
        f'{PLS_COMPONENTS}-component PLS on log10_time_horizon_months',
        'R² (held-out)',
        emphasis=('mean across datasets', mean_target_by_condition[key]),
    ).show()


In [ ]:
if conditions:
    rows = [(name, key) for key in conditions for name in target_datasets if key in results_by_dataset[name]['target_r2']]
    grid = np.stack([
        np.asarray(results_by_dataset[name]['target_r2'][key], dtype=float) for name, key in rows
    ])
    fig = go.Figure(
        go.Heatmap(
            z=grid,
            x=layer_labels,
            y=[f'{name} | {key}' for name, key in rows],
            colorscale=SEQUENTIAL_BLUE,
            zmin=float(np.nanmin(grid)),
            zmax=float(np.nanmax(grid)),
            colorbar={'title': {'text': 'R²', 'side': 'right'}, 'thickness': 14, 'outlinewidth': 0},
            hovertemplate='%{y}<br>layer %{x}<br>R² %{z:.3f}<extra></extra>',
        )
    )
    style_figure(
        fig,
        'Target decodability by dataset, position, split, and layer',
        'Columns bright in every row are the layers worth keeping everywhere',
        height=140 + 26 * len(rows),
    )
    fig.update_xaxes(title_text='Layer')
    fig.show()


In [ ]:
if target_datasets:
    stacked_correlation = np.stack([
        np.abs(results_by_dataset[name]['pls_score_correlation']) for name in target_datasets
    ])
    matrix_figure(
        stacked_correlation.mean(axis=0),
        layer_labels,
        'Mean |correlation| of the first PLS score across layers',
        'The PLS sign is arbitrary per layer, so magnitude carries the meaning: near 1 = identical prompt ordering',
        'mean |r|',
        zmin=0.0,
        zmax=1.0,
    ).show()


### Residual CKA — shared structure that is *not* the horizon

The 6 PLS components are deflated out of every layer and CKA is recomputed. Comparing against
the plain CKA separates two situations: layers that agree only because both encode the horizon
(the block fades), and layers that are wholesale copies of each other (the block survives).


In [ ]:
if target_datasets:
    mean_residual_cka = np.stack([results_by_dataset[name]['residual_cka'] for name in target_datasets]).mean(axis=0)
    matrix_figure(
        mean_residual_cka,
        layer_labels,
        'Linear CKA after removing the 6 PLS components, mean across datasets',
        'Blocks that survive are duplication beyond the time-horizon signal',
        'mean CKA (residual)',
        zmin=float(mean_residual_cka.min()),
        zmax=1.0,
    ).show()

    mean_plain_cka = np.stack([results_by_dataset[name]['cka'] for name in target_datasets]).mean(axis=0)
    matrix_figure(
        mean_plain_cka - mean_residual_cka,
        layer_labels,
        'CKA lost by removing the PLS components',
        'Positive cells: the layers agreed mainly because both carry the horizon signal',
        'ΔCKA',
        diverging=True,
    ).show()


## 11. From matrices to a layer shortlist

The aggregated CKA becomes a distance (`1 - mean CKA`), clustered with average linkage and cut at
`REDUNDANCY_CKA_THRESHOLD`. Each cluster is a set of mutually redundant layers; its
representative is the member with the highest mean target R² under the ranking condition —
leave-task-out at the final position by default, because a layer that only reads out the horizon
for tasks it has seen is the weaker choice.


In [ ]:
distance = np.clip(1.0 - mean_cka, 0.0, None)
np.fill_diagonal(distance, 0.0)
linkage_matrix = linkage(squareform(distance, checks=False), method='average')
cluster_ids = fcluster(linkage_matrix, t=1.0 - REDUNDANCY_CKA_THRESHOLD, criterion='distance')

dendrogram = ff.create_dendrogram(
    np.zeros((len(layers), 1)),
    labels=layer_labels,
    linkagefun=lambda _: linkage_matrix,
    colorscale=[SERIES_COLORS[0]] * 8,
)
style_figure(
    dendrogram,
    'Layer clustering on 1 - mean CKA',
    f'Average linkage over {len(dataset_names)} dataset(s); merges below {1 - REDUNDANCY_CKA_THRESHOLD:.2f} are redundant at the chosen threshold',
    height=440,
)
dendrogram.update_yaxes(title_text='1 - mean CKA')
dendrogram.update_xaxes(title_text='Layer')
dendrogram.show()


In [ ]:
mean_off_diagonal = [float((mean_cka[index].sum() - 1.0) / (len(layers) - 1)) for index in range(len(layers))]
summary = pd.DataFrame({
    'layer': layers,
    'cluster': cluster_ids,
    'mean_cka_with_previous': mean_adjacent,
    'mean_cka_with_others': mean_off_diagonal,
    'cross_dataset_std_cka': [float(std_cka[index].mean()) for index in range(len(layers))],
    'mean_position_cka': mean_position,
    'mean_participation_ratio': mean_participation,
})
for key, values in mean_target_by_condition.items():
    summary[f'mean_target_r2__{key}'] = values
for name in dataset_names:
    summary[f'cka_with_previous__{name}'] = results_by_dataset[name]['adjacent_cka']
for name in target_datasets:
    for key, values in results_by_dataset[name]['target_r2'].items():
        summary[f'target_r2__{name}__{key}'] = values

preferred = [key for key in conditions if key.endswith(RANKING_SPLIT) and key.startswith(f'pos{positions[-1]}')]
fallback = [key for key in conditions if key.startswith(f'pos{positions[-1]}')]
if preferred:
    ranking_column = f'mean_target_r2__{preferred[0]}'
elif fallback:
    ranking_column = f'mean_target_r2__{fallback[0]}'
else:
    ranking_column = 'mean_cka_with_others'
summary['is_cluster_representative'] = (
    summary[ranking_column] == summary.groupby('cluster')[ranking_column].transform('max')
)

representatives = summary.loc[summary['is_cluster_representative'], 'layer'].tolist()
print(f'{summary["cluster"].nunique()} cluster(s) at mean CKA >= {REDUNDANCY_CKA_THRESHOLD}')
print(f'Ranked within clusters by: {ranking_column}')
print(f'Suggested layers to keep: {representatives}')
print(f'Redundant layers to drop: {[layer for layer in layers if layer not in representatives]}')
summary.round(4)


In [ ]:
records = []
for name in dataset_names:
    result = results_by_dataset[name]
    record = {
        'dataset': name,
        'rows': result['row_count'],
        'has_target': result['has_target'],
        'mean_adjacent_cka': float(np.nanmean(result['adjacent_cka'])),
        'min_adjacent_cka': float(np.nanmin(result['adjacent_cka'])),
        'layer_of_min_adjacent_cka': int(layers[int(np.nanargmin(result['adjacent_cka']))]),
        'mean_position_cka': float(np.mean(result['position_cka'])),
        'redundant_layer_pairs': int((result['cka'][triangle] >= REDUNDANCY_CKA_THRESHOLD).sum()),
        'total_layer_pairs': int(len(triangle[0])),
    }
    for split_name, (train_size, test_size) in result['splits'].items():
        record[f'{split_name}_train_rows'] = train_size
        record[f'{split_name}_test_rows'] = test_size
    for key, values in result['target_r2'].items():
        record[f'best_r2__{key}'] = float(np.nanmax(values))
        record[f'best_layer__{key}'] = int(layers[int(np.nanargmax(values))])
    records.append(record)

per_dataset_summary = pd.DataFrame(records)
per_dataset_summary.round(4)


### Write both tables out


In [ ]:
output_dir = repo_root / 'results' / 'layer_redundancy'
output_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(output_dir / 'layer_redundancy_summary.csv', index=False)
per_dataset_summary.to_csv(output_dir / 'per_dataset_summary.csv', index=False)
print(f'Wrote {output_dir / "layer_redundancy_summary.csv"}')
print(f'Wrote {output_dir / "per_dataset_summary.csv"}')


### Clean up the staged arrays

Memmaps are kept between runs so the notebook can be restarted without re-downloading. Run this
cell when finished with them.


In [ ]:
# shutil.rmtree(MEMMAP_DIR, ignore_errors=True)
# shutil.rmtree(BATCH_DIR, ignore_errors=True)
# print(f'Removed {MEMMAP_DIR} and {BATCH_DIR}')


## 12. Reading these matrices

- **Bright square blocks on the mean CKA diagonal with low cross-dataset std** are the headline
  result: contiguous layer runs holding the same geometry in every dataset. Keep one layer per
  block.
- **A block that survives in the residual CKA** is genuine duplication. A block that disappears
  means the layers only agreed through the horizon signal — they may still differ in what else
  they encode.
- **Asymmetry in the predictivity matrix** gives the direction: if layer *i* predicts *j* well but
  not the reverse, *j* is closer to a compression of *i*, so *i* is the one to keep.
- **A large i.i.d. minus leave-task-out gap** at a layer means its horizon readout is largely
  task-specific; prefer layers whose two curves stay close.
- **High mean |r| in the PLS-score correlation with differing target R²** means both layers hold
  the horizon axis but one reads it out more cleanly — prefer the higher R².
- **A high position-CKA curve** says the two cached positions are near-duplicates at that layer,
  so concatenating them doubles the feature width for little gain.
- **Low values in the cross-dataset agreement matrix** are a warning: pruning decisions taken on
  one dataset there will not transfer to the others.
